In [2]:
%pip install sentence-transformers transformers umap-learn


  Using cached sentence_transformers-5.5.0-py3-none-any.whl.metadata (18 kB)
  Using cached transformers-5.8.1-py3-none-any.whl.metadata (33 kB)
  Using cached umap_learn-0.5.12-py3-none-any.whl.metadata (24 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached pynndescent-0.6.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached sentence_transformers-5.5.0-py3-none-any.whl (588 kB)
Using cached transformers-5.8.1-py3-none-any.whl (10.6 MB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)
Using cached umap_learn-0.5.12-py3-none-any.whl (91 kB)
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.8/2.8 MB 7.5 MB/s e


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 03 - Embeddings & Visualization
## Генерація ембедингів та візуалізація (2b, 2c)

У цьому notebook ми:
- Згенеруємо ембединги за допомогою sentence-transformers
- Зменшимо розмірність за допомогою UMAP/t-SNE
- Побудуємо scatter plots
- Збережемо ембединги для навчання моделі

In [3]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.embeddings import TextEmbedder, save_embeddings, load_embeddings
from src.dimensionality import (
    reduce_dimensions,
    plot_embeddings_2d,
    create_embedding_dataframe
)
from src.config import PROCESSED_DATA_PATH, OUTPUTS_PATH, MODEL_NAME

%matplotlib inline

## 1. Завантаження даних

In [4]:
# Завантажуємо оброблені дані
df = pd.read_csv(PROCESSED_DATA_PATH / 'cleaned_data.csv')
print(f"Завантажено {len(df)} записів")
df.head()

Завантажено 274 записів


,text,label,lang,source,link,cleaned_text,processed_text
0,оборона всу в донецкой области осталась без сн...,фейк,rus,Facebook,https://www.facebook.com/permalink.php?story_f...,the financial times ft,financial times ft
1,flash colonel macgregor la contreattaque russe...,правда,fre,Facebook,https://www.facebook.com/watch/?v=826773688331...,flash colonel macgregor la contreattaque russe...,flash colonel macgregor la contreattaque russe...
2,wer kontrolliert die algorithmen die unser leb...,фейк,ger,Facebook,https://www.facebook.com/Wissenschaftswelle/po...,wer kontrolliert die algorithmen die unser leb...,wer kontrolliert die algorithmen die unser leb...
3,quick update from ukraine including some stuff...,правда,eng,Facebook,https://www.facebook.com/TenbyPowellKiwiK.A.R....,quick update from ukraine including some stuff...,quick update ukraine including stuff rugby fir...
4,reel,фейк,ger,Facebook,https://www.facebook.com/reel/6319655381423078,reel,reel


## 2. Генерація ембедингів

In [5]:
# Ініціалізуємо embedder
embedder = TextEmbedder(model_name=MODEL_NAME)
print(f"\nВикористовується модель: {MODEL_NAME}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\marta\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\marta\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded model: sentence-transformers/all-MiniLM-L6-v2

Використовується модель: sentence-transformers/all-MiniLM-L6-v2


In [ ]:
# Генеруємо ембединги
print("Генерація ембедингів...")
embeddings = embedder.encode_dataframe(df, text_column='processed_text')

print(f"\nРозмір ембедингів: {embeddings.shape}")
print(f"Кількість зразків: {embeddings.shape[0]}")
print(f"Розмірність: {embeddings.shape[1]}")

In [ ]:
# Збережемо ембединги
embeddings_path = PROCESSED_DATA_PATH / 'embeddings.npy'
save_embeddings(embeddings, str(embeddings_path))

## 3. Зменшення розмірності за допомогою UMAP

In [ ]:
# Застосуємо UMAP
embeddings_umap = reduce_dimensions(
    embeddings,
    method='umap',
    n_components=2,
    n_neighbors=15,
    min_dist=0.1
)

print(f"UMAP розмірність: {embeddings_umap.shape}")

In [ ]:
# Візуалізуємо UMAP
plot_embeddings_2d(
    embeddings_umap,
    df['label'].values,
    title='UMAP Visualization of News Embeddings',
    save=True,
    filename='embeddings_umap.png'
)

## 4. Зменшення розмірності за допомогою t-SNE

In [ ]:
# Застосуємо t-SNE
embeddings_tsne = reduce_dimensions(
    embeddings,
    method='tsne',
    n_components=2,
    perplexity=30
)

print(f"t-SNE розмірність: {embeddings_tsne.shape}")

In [ ]:
# Візуалізуємо t-SNE
plot_embeddings_2d(
    embeddings_tsne,
    df['label'].values,
    title='t-SNE Visualization of News Embeddings',
    save=True,
    filename='embeddings_tsne.png'
)

## 5. Порівняння методів

In [ ]:
# Порівняємо обидва методи на одному графіку
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# UMAP
scatter1 = axes[0].scatter(
    embeddings_umap[:, 0],
    embeddings_umap[:, 1],
    c=df['label'],
    cmap='viridis',
    alpha=0.6,
    s=10
)
axes[0].set_title('UMAP')
axes[0].set_xlabel('Dimension 1')
axes[0].set_ylabel('Dimension 2')
plt.colorbar(scatter1, ax=axes[0])

# t-SNE
scatter2 = axes[1].scatter(
    embeddings_tsne[:, 0],
    embeddings_tsne[:, 1],
    c=df['label'],
    cmap='viridis',
    alpha=0.6,
    s=10
)
axes[1].set_title('t-SNE')
axes[1].set_xlabel('Dimension 1')
axes[1].set_ylabel('Dimension 2')
plt.colorbar(scatter2, ax=axes[1])

plt.tight_layout()
plt.savefig(OUTPUTS_PATH / 'comparison_umap_tsne.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Збереження результатів

In [ ]:
# Створимо DataFrame з 2D ембедингами
df_umap = create_embedding_dataframe(
    embeddings_umap,
    df['label'].values,
    texts=df['text'].head(100).tolist()  # Зберігаємо лише перші 100 текстів
)

# Збережемо
df_umap.to_csv(PROCESSED_DATA_PATH / 'embeddings_2d_umap.csv', index=False)
print("Збережено 2D ембединги (UMAP)")

## Висновки

✅ Згенеровано ембединги за допомогою sentence-transformers
✅ Застосовано методи зменшення розмірності (UMAP і t-SNE)
✅ Візуалізовано структуру даних у 2D просторі
✅ Збережено ембединги для навчання моделі